# TCT nickel

See the repository README and reproduction guide for data requirements and experiment settings. Generated models and histories are written to `outputs/tct_nickel/`.


In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "research_paths.py").is_file()
)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
from research_paths import data_file, external_file, checkpoint_file, history_file, output_file


In [ ]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv


In [ ]:
"""
LOAD TRAINING DATA
"""

# Load training data
path = data_file("spectra/training_data_long_ontime.csv")

# Read the data from CSV
data = pd.read_csv(path, dtype=float)
col = list(data.columns)

# Extract wavelengths
wavelength = np.array([float(j) for j in col[18:1880]])

# Convert data to numpy array
data = np.array(data)

# Extract specific columns for labels
ontime = data[:, col.index("Ontime")]
conductivity = data[:, col.index("Conductivity")]
concentration = data[:, col.index("Ni")]

# Extract spectrum containing only spectrum information
spectrum = data[:, 18:1880]
spectrum = np.clip(spectrum, None, 60000)

# Indices to be removed
Pb0_index = np.array([col.index("278.249"), col.index("286.893")]) - 18
Pb1_index = np.array([col.index("360.102"), col.index("375.499")]) - 18
Pb2_index = np.array([col.index("400.609"), col.index("410.415")]) - 18
Cu_index = np.array([col.index("323.017"), col.index("340.015")]) - 18
Zn1_index = np.array([col.index("210.444"), col.index("220.089")]) - 18
Zn2_index = np.array([col.index("468.907"), col.index("485.042")]) - 18
# Collect indices to be removed
zero_index = np.concatenate(
    [
        np.arange(Pb0_index[0], Pb0_index[-1] + 1),
        np.arange(Pb1_index[0], Pb1_index[-1] + 1),
        np.arange(Pb2_index[0], Pb2_index[-1] + 1),
        np.arange(Cu_index[0], Cu_index[-1] + 1),
        np.arange(Zn1_index[0], Zn1_index[-1] + 1),
        np.arange(Zn2_index[0], Zn2_index[-1] + 1),
    ]
)

# Set specific indices to zero
spectrum[:, zero_index] = 0
spectrum = spectrum / 60000

# Dimensions and number of spectra
dimension = spectrum.shape[1]
training_number = spectrum.shape[0]

print("\n" + "=" * 40 + "\n")
print("Spectra Dimension - final", dimension)
print("Training Spectra number - final", training_number)
print("Training Data shape", spectrum.shape)
print("Training Label shape", concentration.shape)
print("\n" + "=" * 40 + "\n")


In [ ]:
"""
LOAD TESTING DATA
"""

path = data_file("spectra/testing_data_long_ontime.csv")
# data not in np.array() form (no wavelength)
data_testing = pd.read_csv(path, dtype=float)
# data in np.array() form (no wavelength)
data_testing = np.array(data_testing)
# Ontime
ontime_test = data_testing[:, col.index("Ontime")]
# Conductivity
conductivity_test = data_testing[:, col.index("Conductivity")]
# Concentration labels in ppm (target element selected below).
concentration_test = data_testing[:, col.index("Ni")]
# spectrum_test containing only spectrum information
spectrum_test = data_testing[:, 18:1880]
spectrum_test = np.clip(spectrum_test, None, 60000)
spectrum_test[:, zero_index] = 0
spectrum_test = spectrum_test / 60000
dimension = len(spectrum_test[0])
testing_number = len(spectrum_test)
print("\n" + "=" * 40 + "\n")
print("Sprctra Dimension - final", dimension)
print("Testing Spectra number - final", testing_number)
print("Testing Data shape", spectrum_test.shape)
print("Testing Label shape", concentration_test.shape)
print("\n" + "=" * 40 + "\n")


In [ ]:
plt.plot(wavelength, spectrum[0])
nonzero_count = sum(1 for w in spectrum[0] if w != 0)
print(nonzero_count)


In [ ]:
ontime = ontime.reshape(len(ontime), 1)
conductivity = conductivity.reshape(len(conductivity), 1)
concentration = concentration.reshape(len(concentration), 1)
ontime_test = ontime_test.reshape(len(ontime_test), 1)
conductivity_test = conductivity_test.reshape(len(conductivity_test), 1)
concentration_test = concentration_test.reshape(len(concentration_test), 1)
# Stack the arrays horizontally
training = spectrum
testing = spectrum_test


In [ ]:
import tensorflow.keras as tfkeras
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split
from matplotlib.pyplot import colorbar
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
import numpy as np

# Training parameters
learning_rate = 0.00002
# 0.000001
opt = tfkeras.optimizers.Adam(learning_rate=learning_rate)
# early_stop = tfkeras.callbacks.EarlyStopping(monitor='val_mae', patience=3000,verbose=0, mode='min',restore_best_weights='True')


In [ ]:
## from matplotlib.pyplot import colorbar1
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from keras.layers import LSTM, Dense

dim = int(np.max(ontime) + 1)
input_spectrum = tf.keras.Input(shape=(training.shape[1],))
input_conductivity = tf.keras.Input(shape=(1,))
input_ontime = tf.keras.Input(shape=(1,))

# Reshape input_spectrum
spectrum_shape = layers.Reshape((training.shape[1], 1))(input_spectrum)
conv1 = layers.Conv1D(32, 10, kernel_regularizer=regularizers.l2(0.001))(spectrum_shape)
conv1 = layers.MaxPooling1D(2)(conv1)
conv2 = layers.Conv1D(32, 10, kernel_regularizer=regularizers.l2(0.001))(conv1)
conv2 = layers.MaxPooling1D(2)(conv2)

# Transformer Encoder Block
# Multi-head Attention
attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=64)(conv2, conv2)
attention_output = layers.Dropout(0.1)(attention_output)
attention_output = layers.LayerNormalization(epsilon=1e-6)(attention_output + conv2)  # Add & Norm

# Feed-Forward Network (FFN)
ffn = layers.Dense(16, activation="relu")(attention_output)
ffn = layers.Dropout(0.1)(ffn)
ffn = layers.Dense(32)(ffn)
ffn_output = layers.LayerNormalization(epsilon=1e-6)(ffn + attention_output)  # Add & Norm

# Flatten and Dense Layers for Final Prediction
flatten = layers.Flatten()(ffn_output)
dense = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.001))(flatten)
dense = layers.Dropout(0.2)(dense)
dense = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.001))(dense)
dense = layers.Dropout(0.2)(dense)
output = layers.Dense(1)(dense)

# Build Model
TCT_Model = Model(inputs=input_spectrum, outputs=output, name="Temporal_Convolutional_Transformer")
TCT_Model.summary()


def smape(y_true, y_pred):
    epsilon = tf.keras.backend.epsilon()  # Small constant to avoid division by zero
    numerator = tf.abs(y_pred - y_true)
    denominator = tf.abs(y_true) + tf.abs(y_pred)
    smape_loss = tf.where(
        tf.equal(y_true, 0),
        numerator,  # Use absolute error when y_true is zero
        numerator / (denominator + epsilon) * 2,  # Use SMAPE otherwise
    )
    return tf.reduce_mean(smape_loss) * 100


# Compile and train the model
TCT_Model.compile(loss=smape, optimizer=opt, metrics=["mae"])


In [ ]:
# for layer in CNN_Model.layers:
#    if layer.name.startswith("embed"):
#        transition_layer_name = layer.name
#        break
#    else:
#        continue

# transition_layer = CNN_Model.get_layer(transition_layer_name)

# Embed_Checking = tfkeras.Model(CNN_Model.input[1], transition_layer.output)
# Embed_Checking.summary()

# Embed_Checking.predict()


In [ ]:
# Choose between training and loading a saved model
# Train model ==> 1 / Load model ==> 0
# Train a new model and record its history
X_train, Y_train, X_val, Y_val = train_test_split(
    training, concentration, test_size=0.1, random_state=48
)
train_history = TCT_Model.fit(
    X_train,
    X_val,
    validation_data=(testing, concentration_test),
    batch_size=32,
    epochs=20000,
    verbose=2,
)
# , callbacks=[early_stop]
# Initialize the matrix for training and validation loss and MAE
Epoch = len(train_history.history["mae"])
Training_History = np.zeros((4, Epoch))
Training_History[0, :] = np.array(train_history.history["loss"])
Training_History[1, :] = np.array(train_history.history["mae"])
Training_History[2, :] = np.array(train_history.history["val_loss"])
Training_History[3, :] = np.array(train_history.history["val_mae"])
Array_name = str(output_file("Ni_TCT.npy", "tct_nickel"))
np.save(Array_name, Training_History)
# Save the model
Model_name = str(output_file("Ni_TCT.h5", "tct_nickel"))
# TCT_Model.save(Model_name)


In [ ]:
# Training
Y_pred = TCT_Model.predict([training])
plt.plot(Y_pred, ".")
plt.plot(concentration)
plt.xlabel("number of training data")
plt.ylabel("ppm")
plt.show()
print(tf.keras.losses.MeanSquaredError()(concentration, Y_pred))


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

plt.plot(Training_History[1, :], color="blue", label="Training mae")
plt.plot(Training_History[3, :], color="red", label="Val mae")
plt.legend()
plt.xlabel("epochs")
plt.ylabel("mae")
plt.show()
Y_pred = TCT_Model.predict([testing])
print(tf.keras.losses.MeanSquaredError()(concentration_test, Y_pred))

# Testing
plt.plot(range(0, 30), Y_pred[:30], ".", color="orange", alpha=0.7)
plt.plot(range(30, 60), Y_pred[30:60], ".", color="m", alpha=0.7)
plt.plot(range(60, 90), Y_pred[60:90], ".", color="orange", alpha=0.7)
plt.plot(range(90, 120), Y_pred[90:120], ".", color="m", alpha=0.7)
plt.plot(range(120, 150), Y_pred[120:150], ".", color="orange", alpha=0.7)
plt.plot(range(150, 180), Y_pred[150:180], ".", color="m", alpha=0.7)
plt.plot(range(180, 210), Y_pred[180:210], ".", color="orange", alpha=0.7)
plt.plot(range(210, 240), Y_pred[210:240], ".", color="m", alpha=0.7)
plt.plot(range(240, 270), Y_pred[240:270], ".", color="orange", alpha=0.7)
plt.plot(range(270, 300), Y_pred[270:300], ".", color="m", alpha=0.7)
plt.ylim(0, 22)
plt.plot(concentration_test, color="black", linewidth=3.0)
plt.xlabel("number of testing data", fontsize=12)
plt.ylabel("Concentration (ppm)", fontsize=12)
plt.title("Predictions vs Actual Concentrations", fontsize=14)

# Add legend
orange_patch = mpatches.Patch(color="orange", label="7.5")
magenta_patch = mpatches.Patch(color="m", label="15")
plt.legend(handles=[orange_patch, magenta_patch], fontsize=10, loc="upper right")
# Add arrow pointing to the right
plt.annotate(
    "", xy=(300, 1), xytext=(0, 1), arrowprops=dict(facecolor="black", arrowstyle="->", linewidth=3)
)
plt.text(150, 2, "Conductivity Increases", fontsize=16, ha="center")
plt.show()
Y_pred = TCT_Model.predict([testing])
print(tf.keras.losses.MeanSquaredError()(concentration_test, Y_pred))


In [ ]:
mae = tf.keras.losses.MeanAbsoluteError()
print(mae(concentration_test, Y_pred).numpy())


In [ ]:
Y_pred = TCT_Model.predict([testing])
np.mean(Y_pred)
